In [ ]:
import os
import urllib.request
from urllib.error import HTTPError
from tokenizers import ByteLevelBPETokenizer
import time
import numpy as np
import multiprocessing as mp
import tensorflow as tf
import matplotlib.pyplot as plt

# Data exploration

We will look into the distribution of the number of tokens.
As a prerequisite, we need to:
  - download the dataset
  - Build the BPE tokenizer

Let start with downloading the dataset.

In [ ]:
!rm data.py configs.py tokenizer.py
!rm -rf de_tokenizer_30_000_vocab_size_model
!rm -rf en_tokenizer_30_000_vocab_size_model

In [ ]:
base_url = "https://raw.githubusercontent.com/MiguelSteph/transformer-from-scratch/main/"
def download_file_from_github(file_path: str, file_name: str):
    if not os.path.isfile(file_name):
        file_url = base_url + file_path
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_name)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file directly from the GitHub repository:\n", e)

file_paths = [
    'configs/configs.py',
    'data/tokenizer.py',
    'data/data.py'
]
file_names = [
    'configs.py',
    'tokenizer.py',
    'data.py'
]

for file_path, file_name in zip(file_paths, file_names): 
    download_file_from_github(file_path, file_name)

In [ ]:
from configs import get_configs
from data import build_and_save_tokenizer_models, download_and_get_raw_datasets

config = get_configs()

# Build and save BPE tokenizer model

This is one time script that will create the BPE tokenizer model and dave it.

In [ ]:
build_and_save_tokenizer_models()

# Data exploration

In [ ]:
train_ds, validation_ds, test_ds = download_and_get_raw_datasets()
full_ds = train_ds.concatenate(validation_ds).concatenate(test_ds)
de_data = full_ds.map(lambda x: x["de"], num_parallel_calls=tf.data.AUTOTUNE)
en_data = full_ds.map(lambda x: x["en"], num_parallel_calls=tf.data.AUTOTUNE)

de_tokenizer = ByteLevelBPETokenizer(config.data.de_tokenizer_model_path + '/vocab.json',
                                     config.data.de_tokenizer_model_path + '/merges.txt')
de_tokenizer.add_special_tokens(list(config.data.special_tokens))
en_tokenizer = ByteLevelBPETokenizer(config.data.en_tokenizer_model_path + '/vocab.json',
                                     config.data.en_tokenizer_model_path + '/merges.txt')
en_tokenizer.add_special_tokens(list(config.data.special_tokens))

def get_en_tokens_length(input: np.ndarray) -> list[int]:
    lengths = []
    for sample in input:
        lengths.append(len(en_tokenizer.encode(sample.decode('utf-8')).ids))
    return lengths

def get_de_tokens_length(input: np.ndarray) -> list[int]:
    lengths = []
    for sample in input:
        lengths.append(len(de_tokenizer.encode(sample.decode('utf-8')).ids))
    return lengths

start_time = time.time()
cpu_count = os.cpu_count()
with mp.Pool(cpu_count) as pool:
    de_tokens_lengths = []
    for tokens_length in pool.imap(get_de_tokens_length, de_data.batch(100).as_numpy_iterator(), chunksize=16):
        de_tokens_lengths.extend(tokens_length)

print(f"German Token length counting took {time.time() - start_time}")


start_time = time.time()
cpu_count = os.cpu_count()
with mp.Pool(cpu_count) as pool:
    en_tokens_lengths = []
    for tokens_length in pool.imap(get_en_tokens_length, en_data.batch(100).as_numpy_iterator(), chunksize=16):
        en_tokens_lengths.extend(tokens_length)

print(f"English Token length counting took {time.time() - start_time}")

In [ ]:
print(f"Max token length for german sentences is {max(de_tokens_lengths)}")
print(f"Max token length for english sentences is {max(en_tokens_lengths)}")
print()
print(f"Min token length for german sentences is {min(de_tokens_lengths)}")
print(f"Min token length for english sentences is {min(en_tokens_lengths)}")

plt.hist([en_tokens_lengths, de_tokens_lengths], color=['r', 'b'], label=['english', 'german'], alpha=0.5)
plt.legend()
plt.show()

In [ ]:
max_tokens = 100
de_out_of_range_samples = sum(item > max_tokens for item in de_tokens_lengths)
en_out_of_range_samples = sum(item > max_tokens for item in en_tokens_lengths)
print(f"With max_seq_len set to {max_tokens}, we have {de_out_of_range_samples} german samples sentences out of range.")
print(f"With max_seq_len set to {max_tokens}, we have {en_out_of_range_samples} german samples sentences out of range.")

de_reduced_tokens_lengths = [token_len for token_len in de_tokens_lengths if token_len <= max_tokens]
en_reduced_tokens_lengths = [token_len for token_len in en_tokens_lengths if token_len <= max_tokens]

plt.hist([en_reduced_tokens_lengths, de_reduced_tokens_lengths], color=['r', 'b'], label=['english', 'german'], alpha=0.5)
plt.legend()
plt.show()
